# Version test de gradio

In [ ]:
import gradio as gr

# =============================
# = Fonctions factices (stubs)=
# =============================
def generate_chatbot_response(user_input, chat_history):
    return f"[Chatbot fictif] J'ai bien reçu: {user_input}"

def translate_text_to_en(text):
    return f"[Fake Translation in English] {text}"

def analyze_sentiment(text):
    text_lower = text.lower()
    if "bien" in text_lower and "pas bien" not in text_lower:
        return "positif"
    elif "pas bien" in text_lower:
        return "négatif"
    else:
        return "neutre"

# ============================
# = Partie "pipeline" Gradio =
# ============================
def conversation_pipeline(user_input, history):
    user_msg = user_input
    bot_msg = f"Réponse au message utilisateur"

    user_msg_en = f"[Fake Translation in English] {user_msg}"
    bot_msg_en = f"[Fake Translation in English] {bot_msg}"

    user_sentiment = analyze_sentiment(user_msg)
    bot_sentiment = analyze_sentiment(bot_msg)

    user_side = {
        "original": user_msg,
        "english": user_msg_en,
        "sentiment": user_sentiment
    }
    bot_side = {
        "original": bot_msg,
        "english": bot_msg_en,
        "sentiment": bot_sentiment
    }

    history = history + [(user_side, bot_side)]
    return history, bot_msg


def format_conversation_display_as_messages(history):
    """
    Convertit l'historique en une liste de dicts:
      [{"role": "user"|"assistant", "content": "..."}]
    
    On veut trois lignes, séparées par des retours à la ligne :
      1) Le texte en français
      2) Le texte en anglais
      3) Le sentiment
    """
    messages = []
    for (user_side, bot_side) in history:
        # Bloc de l'utilisateur
        user_content = (
            f"{user_side['original']}\n\n"
            f"{user_side['english']}\n\n"
            f"Sentiment : {user_side['sentiment']}"
        )
        messages.append({
            "role": "user",
            "content": user_content
        })
        
        # Bloc du chatbot
        bot_content = (
            f"{bot_side['original']}\n\n"
            f"{bot_side['english']}\n\n"
            f"Sentiment : {bot_side['sentiment']}"
        )
        messages.append({
            "role": "assistant",
            "content": bot_content
        })
    return messages

def on_user_submit(user_msg, history):
    new_history, bot_msg = conversation_pipeline(user_msg, history)
    messages = format_conversation_display_as_messages(new_history)
    return new_history, messages

# ---------------------------------------
# Ajout d'un CSS personnalisé
# ---------------------------------------
MY_CUSTOM_CSS = """
.user.message .message-content {
    background-color: #6C2BD9 !important; /* violet flash */
    color: white !important;
}

.bot.message .message-content {
    background-color: #34C759 !important; /* vert */
    color: white !important;
}
"""

with gr.Blocks(css=MY_CUSTOM_CSS) as demo:
    gr.Markdown("# Chatbot avec traduction FR/EN et analyse de sentiment")
    conversation_state = gr.State([])

    # Composant Chatbot en mode "messages"
    conversation_display = gr.Chatbot(
        label="Historique de la conversation",
        type="messages"
    )
    
    user_input = gr.Textbox(label="Votre message")

    send_button = gr.Button("Envoyer")

    send_button.click(
        fn=on_user_submit,
        inputs=[user_input, conversation_state],
        outputs=[conversation_state, conversation_display]
    )

    user_input.submit(
        fn=on_user_submit,
        inputs=[user_input, conversation_state],
        outputs=[conversation_state, conversation_display]
    )

# Lancement local
if __name__ == "__main__":
    demo.launch()